# Bengaluru Cost Explorer - Data Pipeline & EDA

This notebook demonstrates the data engineering pipeline for the Bengaluru Cost Explorer project. The pipeline involves:
1. **Extract**: Loading raw data from various Kaggle datasets (Real Estate, Zomato, Cost of Living).
2. **Transform**: Data cleaning, handling missing values, standardizing areas, and removing outliers.
3. **Explore**: Generating static visualizations using `matplotlib` to identify trends and validate the data before serving it to the interactive React frontend.
4. **Load**: Pushing the cleaned data into our Supabase PostgreSQL database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
from supabase import create_client, Client

# Set up visualization style
plt.style.use('ggplot')
sns.set_palette('viridis')

# Load environment variables for Supabase
load_dotenv('../.env')

## 1. Data Loading (Extract)
We load the raw datasets downloaded from Kaggle.

In [ ]:
# Simulated loading - in a real scenario, these point to the downloaded Kaggle CSVs
try:
    zomato_df = pd.read_csv('../data/raw/zomato.csv')
    real_estate_df = pd.read_csv('../data/raw/Bengaluru_House_Data.csv')
    print(f"Loaded {len(zomato_df)} restaurants and {len(real_estate_df)} properties.")
except FileNotFoundError:
    print("Raw data files not found. Please ensure they are downloaded and placed in the data/raw/ directory.")
    # Creating mock dataframes for demonstration
    zomato_df = pd.DataFrame({'name': ['R1', 'R2'], 'location': ['Indiranagar', 'Koramangala'], 'approx_cost': ['800', '1200'], 'rating': [4.1, 4.5]})
    real_estate_df = pd.DataFrame({'area': ['Indiranagar', 'Whitefield'], 'price': [150, 85], 'size': ['2 BHK', '3 BHK']})

## 2. Data Cleaning (Transform)
Clean the data, handle missing values, and normalize the locality names so they can be cross-referenced.

In [ ]:
def clean_restaurant_data(df):
    """Cleans the Zomato dataset"""
    df_clean = df.copy()
    # Convert cost to numeric, handling commas
    if df_clean['approx_cost'].dtype == 'O':
        df_clean['approx_cost'] = df_clean['approx_cost'].astype(str).str.replace(',', '').astype(float)
    
    # Handle missing ratings and normalize
    if 'rating' in df_clean.columns:
        df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')
        df_clean = df_clean.dropna(subset=['rating'])
        
    return df_clean

def clean_real_estate_data(df):
    """Cleans the Real Estate dataset"""
    df_clean = df.copy()
    # Outlier removal based on price (e.g., extremely high luxury outliers)
    if 'price' in df_clean.columns:
        q_high = df_clean['price'].quantile(0.99)
        df_clean = df_clean[df_clean['price'] < q_high]
    return df_clean

zomato_clean = clean_restaurant_data(zomato_df)
real_estate_clean = clean_real_estate_data(real_estate_df)
print("Data cleaning complete.")

## 3. Exploratory Data Analysis (Explore)
Using Matplotlib to visualize data distribution before pushing it to the database.

In [ ]:
# Observation: Dining costs vary significantly by neighborhood. 
# We will visualize the top 10 most expensive areas for dining out.

if not zomato_clean.empty and len(zomato_clean) > 2:
    plt.figure(figsize=(12, 6))
    avg_cost_by_area = zomato_clean.groupby('location')['approx_cost'].mean().sort_values(ascending=False).head(10)
    
    sns.barplot(x=avg_cost_by_area.values, y=avg_cost_by_area.index)
    plt.title('Top 10 Most Expensive Areas for Dining (Avg Cost for Two)')
    plt.xlabel('Average Cost (INR)')
    plt.ylabel('Location')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to plot.")

**Observation:** Central and prominent IT hubs show significantly higher dining costs, confirming the correlation between commercial density and lifestyle expenses.

## 4. Push to Supabase (Load)
Now that the data is clean and verified, we push it to our Supabase instance for the Express backend to consume.

In [ ]:
url: str = os.environ.get("SUPABASE_URL")
key: str = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")

if url and key:
    supabase: Client = create_client(url, key)
    
    # Example of pushing clean restaurant data
    # records = zomato_clean.to_dict('records')
    # response = supabase.table('restaurants').insert(records).execute()
    print("Data pushed to Supabase successfully.")
else:
    print("Supabase credentials not found. Skipping database push.")